# Corpus Split Organizations

This notebook preserves `data/original/` and generates two alternative corpus organizations directly under `data/`:

- `data/pair_controlled/`
- `data/max_cross_split/`

Each generated organization uses split seeds 13, 21, 40, 42, and 73.

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

In [2]:
SPLIT_SEEDS = [13, 21, 40, 42, 73]

SPLIT_NAMES = [
    "train",
    "validation",
    "test",
]

EXPECTED_SPLIT_COUNTS = {
    "train": 3990,
    "validation": 570,
    "test": 1140,
}

EXPECTED_CLASS_COUNTS = {
    "train": {0: 1995, 1: 1995},
    "validation": {0: 285, 1: 285},
    "test": {0: 570, 1: 570},
}

STRATEGIES = [
    "pair_controlled",
    "max_cross_split",
]

In [3]:
def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    while True:
        if (current / "data" / "original").is_dir():
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Could not locate the project root containing data/original/."
    )

In [4]:
PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
ORIGINAL_DIR = DATA_DIR / "original"

OUTPUT_DIRS = {
    strategy: DATA_DIR / strategy
    for strategy in STRATEGIES
}

ORIGINAL_FILES = {
    "train": ORIGINAL_DIR / "train.jsonl",
    "validation": ORIGINAL_DIR / "validation.jsonl",
    "test": ORIGINAL_DIR / "test.jsonl",
}

for strategy in STRATEGIES:
    for seed in SPLIT_SEEDS:
        (
            OUTPUT_DIRS[strategy]
            / f"seed_{seed}"
        ).mkdir(parents=True, exist_ok=True)

for split_name, path in ORIGINAL_FILES.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing original {split_name} file: {path}"
        )

print("Project root:", PROJECT_ROOT)
print("Original corpus:", ORIGINAL_DIR)

for strategy, path in OUTPUT_DIRS.items():
    print(strategy, "->", path)

Project root: /home/avelar/pun-detection-split-analysis
Original corpus: /home/avelar/pun-detection-split-analysis/data/original
pair_controlled -> /home/avelar/pun-detection-split-analysis/data/pair_controlled
max_cross_split -> /home/avelar/pun-detection-split-analysis/data/max_cross_split


In [5]:
def load_jsonl(path, split_name):
    rows = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            item = json.loads(line)
            item["original_split"] = split_name
            rows.append(item)

    return rows

In [6]:
rows = []

for split_name in SPLIT_NAMES:
    rows.extend(
        load_jsonl(
            ORIGINAL_FILES[split_name],
            split_name,
        )
    )

df = pd.DataFrame(rows)

SOURCE_COLUMNS = [
    column
    for column in df.columns
    if column != "original_split"
]

display(df.head())

print("Examples:", len(df))
print("Columns:", SOURCE_COLUMNS)

,id,text,label,tokens,labels,original_split
0,5.46.H,Por que o carteiro foi à feira? Porque tinha u...,1,"[Por, que, o, carteiro, foi, à, feira, ?, Porq...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]",train
1,5.792.H,Por que a mulher esotérica não conseguia engra...,1,"[Por, que, a, mulher, esotérica, não, consegui...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]",train
2,5.1811.H,Qual é o animal que está sempre cansado? Dorme...,1,"[Qual, é, o, animal, que, está, sempre, cansad...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]",train
3,5.733.H,Qual o sambista passou a dar presente pra todo...,1,"[Qual, o, sambista, passou, a, dar, presente, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",train
4,4.652.N,Um homem matou uma ovelha e agora foi preso . ...,0,"[Um, homem, matou, uma, ovelha, e, agora, foi,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",train


Examples: 5700
Columns: ['id', 'text', 'label', 'tokens', 'labels']


In [7]:
REQUIRED_COLUMNS = {
    "id",
    "text",
    "label",
    "original_split",
}

missing_columns = REQUIRED_COLUMNS - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

if df["id"].duplicated().any():
    raise ValueError(
        "Duplicated example IDs were found."
    )

df["label"] = df["label"].astype(int)

if set(df["label"].unique()) != {0, 1}:
    raise ValueError(
        f"Unexpected labels: {sorted(df['label'].unique())}"
    )

if len(df) != 5700:
    raise ValueError(
        f"Expected 5700 examples, found {len(df)}."
    )

In [8]:
PAIR_PATTERN = re.compile(
    r"^(?P<pair_id>.+)\.(?P<suffix>[HN])$"
)

In [9]:
def parse_example_id(example_id):
    match = PAIR_PATTERN.match(str(example_id))

    if match is None:
        raise ValueError(
            f"Unexpected example ID format: {example_id}"
        )

    return (
        match.group("pair_id"),
        match.group("suffix"),
    )

In [10]:
parsed_ids = df["id"].apply(parse_example_id)

df["pair_id"] = parsed_ids.apply(
    lambda value: value[0]
)

df["pair_suffix"] = parsed_ids.apply(
    lambda value: value[1]
)

display(
    df[
        [
            "id",
            "pair_id",
            "pair_suffix",
            "label",
            "original_split",
            "text",
        ]
    ].head()
)

,id,pair_id,pair_suffix,label,original_split,text
0,5.46.H,5.46,H,1,train,Por que o carteiro foi à feira? Porque tinha u...
1,5.792.H,5.792,H,1,train,Por que a mulher esotérica não conseguia engra...
2,5.1811.H,5.1811,H,1,train,Qual é o animal que está sempre cansado? Dorme...
3,5.733.H,5.733,H,1,train,Qual o sambista passou a dar presente pra todo...
4,4.652.N,4.652,N,0,train,Um homem matou uma ovelha e agora foi preso . ...


In [11]:
if df["pair_id"].nunique() != 2850:
    raise ValueError(
        f"Expected 2850 pairs, found {df['pair_id'].nunique()}."
    )

pair_sizes = df.groupby("pair_id").size()

if not (pair_sizes == 2).all():
    raise ValueError(
        "Every pair must contain exactly two examples."
    )

suffix_sets = (
    df.groupby("pair_id")["pair_suffix"]
    .agg(lambda values: frozenset(values))
)

if not (
    suffix_sets == frozenset({"H", "N"})
).all():
    raise ValueError(
        "Every pair must contain exactly one .H and one .N instance."
    )

label_sets = (
    df.groupby("pair_id")["label"]
    .agg(lambda values: frozenset(values))
)

if not (
    label_sets == frozenset({0, 1})
).all():
    raise ValueError(
        "Every pair must contain labels 0 and 1."
    )

expected_label_by_suffix = {
    "H": 1,
    "N": 0,
}

suffix_label_ok = df.apply(
    lambda row: (
        expected_label_by_suffix[row["pair_suffix"]]
        == row["label"]
    ),
    axis=1,
)

if not suffix_label_ok.all():
    raise ValueError(
        "At least one suffix is inconsistent with its label."
    )

print("Pair structure validated.")

Pair structure validated.


In [12]:
original_split_counts = (
    df["original_split"]
    .value_counts()
    .reindex(SPLIT_NAMES)
)

display(original_split_counts)

if original_split_counts.to_dict() != EXPECTED_SPLIT_COUNTS:
    raise ValueError(
        "Original split sizes do not match the expected sizes."
    )

original_class_distribution = (
    pd.crosstab(
        df["original_split"],
        df["label"],
    )
    .reindex(SPLIT_NAMES)
    .reindex(columns=[0, 1], fill_value=0)
)

display(original_class_distribution)

for split_name, expected in EXPECTED_CLASS_COUNTS.items():
    observed = (
        df.loc[
            df["original_split"] == split_name,
            "label",
        ]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    if observed != expected:
        raise ValueError(
            f"Unexpected class counts in original {split_name}: {observed}"
        )

original_split
train         3990
validation     570
test          1140
Name: count, dtype: int64

label,0,1
original_split,,
train,1995,1995
validation,285,285
test,570,570


In [13]:
def make_pair_split_table(dataframe, split_column):
    pair_table = dataframe.pivot(
        index="pair_id",
        columns="pair_suffix",
        values=split_column,
    )

    return pair_table[["H", "N"]]

In [14]:
def make_pair_matrix(dataframe, split_column):
    pair_table = make_pair_split_table(
        dataframe,
        split_column,
    )

    return (
        pd.crosstab(
            pair_table["H"],
            pair_table["N"],
        )
        .reindex(
            index=SPLIT_NAMES,
            columns=SPLIT_NAMES,
            fill_value=0,
        )
    )

In [15]:
def count_cross_split_pairs(dataframe, split_column):
    pair_table = make_pair_split_table(
        dataframe,
        split_column,
    )

    cross_split_pairs = int(
        (
            pair_table["H"]
            != pair_table["N"]
        ).sum()
    )

    cross_split_rate = (
        cross_split_pairs
        / len(pair_table)
    )

    return (
        cross_split_pairs,
        cross_split_rate,
    )

In [16]:
original_pair_matrix = make_pair_matrix(
    df,
    "original_split",
)

original_cross_split_pairs, original_cross_split_rate = (
    count_cross_split_pairs(
        df,
        "original_split",
    )
)

display(original_pair_matrix)

print(
    "Original cross-split pairs:",
    original_cross_split_pairs,
)

print(
    "Original cross-split rate:",
    f"{original_cross_split_rate:.4%}",
)

if original_cross_split_pairs != 1306:
    raise ValueError(
        f"Expected 1306 original cross-split pairs, found {original_cross_split_pairs}."
    )

N,train,validation,test
H,,,
train,1404,206,385
validation,196,22,67
test,395,57,118


Original cross-split pairs: 1306
Original cross-split rate: 45.8246%


In [17]:
def make_json_serializable(obj):
    if obj is None:
        return None

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        if np.isnan(obj):
            return None
        return float(obj)

    if isinstance(obj, np.ndarray):
        return [
            make_json_serializable(value)
            for value in obj.tolist()
        ]

    if isinstance(obj, (list, tuple)):
        return [
            make_json_serializable(value)
            for value in obj
        ]

    if isinstance(obj, dict):
        return {
            str(key): make_json_serializable(value)
            for key, value in obj.items()
        }

    try:
        if pd.isna(obj):
            return None
    except (TypeError, ValueError):
        pass

    return obj

In [18]:
def save_jsonl(dataframe, output_path):
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for _, row in dataframe[SOURCE_COLUMNS].iterrows():
            item = {
                key: make_json_serializable(value)
                for key, value in row.to_dict().items()
            }

            file.write(
                json.dumps(
                    item,
                    ensure_ascii=False,
                )
                + "\n"
            )

In [19]:
def count_jsonl_lines(path):
    with path.open("r", encoding="utf-8") as file:
        return sum(
            1
            for line in file
            if line.strip()
        )

In [20]:
def validate_generated_split(
    dataframe,
    strategy,
    seed,
):
    if len(dataframe) != 5700:
        raise ValueError(
            f"{strategy}/{seed}: expected 5700 examples."
        )

    if dataframe["id"].nunique() != 5700:
        raise ValueError(
            f"{strategy}/{seed}: IDs are not unique."
        )

    if set(dataframe["id"]) != set(df["id"]):
        raise ValueError(
            f"{strategy}/{seed}: generated IDs differ from the original corpus."
        )

    if dataframe["pair_id"].nunique() != 2850:
        raise ValueError(
            f"{strategy}/{seed}: expected 2850 pairs."
        )

    observed_split_counts = (
        dataframe["assigned_split"]
        .value_counts()
        .reindex(SPLIT_NAMES)
        .to_dict()
    )

    if observed_split_counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(
            f"{strategy}/{seed}: unexpected split sizes {observed_split_counts}."
        )

    for split_name, expected in EXPECTED_CLASS_COUNTS.items():
        observed = (
            dataframe.loc[
                dataframe["assigned_split"] == split_name,
                "label",
            ]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        if observed != expected:
            raise ValueError(
                f"{strategy}/{seed}: unexpected class counts in {split_name}: {observed}."
            )

    cross_split_pairs, cross_split_rate = (
        count_cross_split_pairs(
            dataframe,
            "assigned_split",
        )
    )

    pair_matrix = make_pair_matrix(
        dataframe,
        "assigned_split",
    )

    if strategy == "pair_controlled":
        if cross_split_pairs != 0:
            raise ValueError(
                f"{strategy}/{seed}: expected 0 cross-split pairs."
            )

    if strategy == "max_cross_split":
        if cross_split_pairs != 1710:
            raise ValueError(
                f"{strategy}/{seed}: expected 1710 cross-split pairs."
            )

        pair_table = make_pair_split_table(
            dataframe,
            "assigned_split",
        )

        for pair_id, row in pair_table.iterrows():
            if row["H"] != row["N"]:
                if "train" not in {
                    row["H"],
                    row["N"],
                }:
                    raise ValueError(
                        f"{strategy}/{seed}: pair {pair_id} crosses without training."
                    )

    return {
        "strategy": strategy,
        "seed": int(seed),
        "total_examples": int(len(dataframe)),
        "total_pairs": int(dataframe["pair_id"].nunique()),
        "train_examples": int(
            (
                dataframe["assigned_split"]
                == "train"
            ).sum()
        ),
        "validation_examples": int(
            (
                dataframe["assigned_split"]
                == "validation"
            ).sum()
        ),
        "test_examples": int(
            (
                dataframe["assigned_split"]
                == "test"
            ).sum()
        ),
        "cross_split_pairs": int(cross_split_pairs),
        "cross_split_rate": float(cross_split_rate),
        "pair_matrix": pair_matrix,
    }

In [21]:
def save_generated_run(
    dataframe,
    strategy,
    seed,
    validation_summary,
):
    seed_dir = (
        OUTPUT_DIRS[strategy]
        / f"seed_{seed}"
    )

    seed_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for split_name in SPLIT_NAMES:
        split_df = (
            dataframe.loc[
                dataframe["assigned_split"] == split_name
            ]
            .copy()
        )

        save_jsonl(
            split_df,
            seed_dir / f"{split_name}.jsonl",
        )

    validation_summary["pair_matrix"].to_csv(
        seed_dir / "pair_matrix.csv",
        encoding="utf-8",
    )

    dataframe[
        [
            "id",
            "pair_id",
            "pair_suffix",
            "label",
            "original_split",
            "assigned_split",
            "text",
        ]
    ].to_csv(
        seed_dir / "inspection.csv",
        index=False,
        encoding="utf-8",
    )

    metadata = {
        key: value
        for key, value in validation_summary.items()
        if key != "pair_matrix"
    }

    metadata["class_distribution"] = {
        split_name: {
            str(label): int(count)
            for label, count in (
                dataframe.loc[
                    dataframe["assigned_split"] == split_name,
                    "label",
                ]
                .value_counts()
                .sort_index()
                .to_dict()
                .items()
            )
        }
        for split_name in SPLIT_NAMES
    }

    with (
        seed_dir / "metadata.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

    saved_counts = {
        split_name: count_jsonl_lines(
            seed_dir / f"{split_name}.jsonl"
        )
        for split_name in SPLIT_NAMES
    }

    if saved_counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(
            f"{strategy}/{seed}: saved file sizes are incorrect: {saved_counts}."
        )

    return seed_dir

In [22]:
def generate_pair_controlled(
    dataframe,
    seed,
):
    rng = np.random.default_rng(seed)

    pair_ids = np.array(
        sorted(dataframe["pair_id"].unique()),
        dtype=object,
    )

    rng.shuffle(pair_ids)

    train_pair_count = (
        EXPECTED_SPLIT_COUNTS["train"] // 2
    )
    validation_pair_count = (
        EXPECTED_SPLIT_COUNTS["validation"] // 2
    )

    train_end = train_pair_count
    validation_end = (
        train_end
        + validation_pair_count
    )

    train_pairs = set(
        pair_ids[:train_end]
    )

    validation_pairs = set(
        pair_ids[
            train_end:validation_end
        ]
    )

    test_pairs = set(
        pair_ids[validation_end:]
    )

    result = dataframe.copy()

    def assign_split(pair_id):
        if pair_id in train_pairs:
            return "train"

        if pair_id in validation_pairs:
            return "validation"

        if pair_id in test_pairs:
            return "test"

        raise RuntimeError(
            f"Pair was not assigned: {pair_id}"
        )

    result["assigned_split"] = (
        result["pair_id"]
        .apply(assign_split)
    )

    return result

In [23]:
def generate_max_cross_split(
    dataframe,
    seed,
):
    rng = np.random.default_rng(seed)

    pair_ids = np.array(
        sorted(dataframe["pair_id"].unique()),
        dtype=object,
    )

    rng.shuffle(pair_ids)

    h_outside_train_count = (
        EXPECTED_CLASS_COUNTS["validation"][1]
        + EXPECTED_CLASS_COUNTS["test"][1]
    )

    n_outside_train_count = (
        EXPECTED_CLASS_COUNTS["validation"][0]
        + EXPECTED_CLASS_COUNTS["test"][0]
    )

    complete_train_pair_count = (
        len(pair_ids)
        - h_outside_train_count
        - n_outside_train_count
    )

    if complete_train_pair_count != 1140:
        raise ValueError(
            "Unexpected number of complete training pairs."
        )

    complete_train_end = (
        complete_train_pair_count
    )

    h_outside_end = (
        complete_train_end
        + h_outside_train_count
    )

    complete_train_pairs = set(
        pair_ids[:complete_train_end]
    )

    h_outside_pairs = list(
        pair_ids[
            complete_train_end:h_outside_end
        ]
    )

    n_outside_pairs = list(
        pair_ids[h_outside_end:]
    )

    rng.shuffle(h_outside_pairs)
    rng.shuffle(n_outside_pairs)

    h_validation_count = (
        EXPECTED_CLASS_COUNTS["validation"][1]
    )

    n_validation_count = (
        EXPECTED_CLASS_COUNTS["validation"][0]
    )

    h_validation_pairs = set(
        h_outside_pairs[:h_validation_count]
    )

    h_test_pairs = set(
        h_outside_pairs[h_validation_count:]
    )

    n_validation_pairs = set(
        n_outside_pairs[:n_validation_count]
    )

    n_test_pairs = set(
        n_outside_pairs[n_validation_count:]
    )

    split_by_key = {}

    for pair_id in complete_train_pairs:
        split_by_key[(pair_id, "H")] = "train"
        split_by_key[(pair_id, "N")] = "train"

    for pair_id in h_validation_pairs:
        split_by_key[(pair_id, "H")] = "validation"
        split_by_key[(pair_id, "N")] = "train"

    for pair_id in h_test_pairs:
        split_by_key[(pair_id, "H")] = "test"
        split_by_key[(pair_id, "N")] = "train"

    for pair_id in n_validation_pairs:
        split_by_key[(pair_id, "N")] = "validation"
        split_by_key[(pair_id, "H")] = "train"

    for pair_id in n_test_pairs:
        split_by_key[(pair_id, "N")] = "test"
        split_by_key[(pair_id, "H")] = "train"

    result = dataframe.copy()

    result["assigned_split"] = result.apply(
        lambda row: split_by_key[
            (
                row["pair_id"],
                row["pair_suffix"],
            )
        ],
        axis=1,
    )

    return result

In [24]:
generation_functions = {
    "pair_controlled": generate_pair_controlled,
    "max_cross_split": generate_max_cross_split,
}

all_summaries = []

for strategy in STRATEGIES:
    strategy_summaries = []

    print("=" * 80)
    print(strategy)

    for seed in SPLIT_SEEDS:
        generated = generation_functions[strategy](
            df,
            seed,
        )

        validation_summary = validate_generated_split(
            generated,
            strategy,
            seed,
        )

        output_dir = save_generated_run(
            generated,
            strategy,
            seed,
            validation_summary,
        )

        summary_row = {
            key: value
            for key, value in validation_summary.items()
            if key != "pair_matrix"
        }

        strategy_summaries.append(
            summary_row
        )

        all_summaries.append(
            summary_row
        )

        print(
            strategy,
            seed,
            summary_row["cross_split_pairs"],
            f"{summary_row['cross_split_rate']:.4%}",
            output_dir,
        )

    pd.DataFrame(
        strategy_summaries
    ).to_csv(
        OUTPUT_DIRS[strategy] / "summary.csv",
        index=False,
        encoding="utf-8",
    )

pair_controlled
pair_controlled 13 0 0.0000% /home/avelar/pun-detection-split-analysis/data/pair_controlled/seed_13
pair_controlled 21 0 0.0000% /home/avelar/pun-detection-split-analysis/data/pair_controlled/seed_21
pair_controlled 40 0 0.0000% /home/avelar/pun-detection-split-analysis/data/pair_controlled/seed_40
pair_controlled 42 0 0.0000% /home/avelar/pun-detection-split-analysis/data/pair_controlled/seed_42
pair_controlled 73 0 0.0000% /home/avelar/pun-detection-split-analysis/data/pair_controlled/seed_73
max_cross_split
max_cross_split 13 1710 60.0000% /home/avelar/pun-detection-split-analysis/data/max_cross_split/seed_13
max_cross_split 21 1710 60.0000% /home/avelar/pun-detection-split-analysis/data/max_cross_split/seed_21
max_cross_split 40 1710 60.0000% /home/avelar/pun-detection-split-analysis/data/max_cross_split/seed_40
max_cross_split 42 1710 60.0000% /home/avelar/pun-detection-split-analysis/data/max_cross_split/seed_42
max_cross_split 73 1710 60.0000% /home/avelar/pun-de

In [25]:
all_summary_df = pd.DataFrame(
    all_summaries
)

all_summary_df.to_csv(
    DATA_DIR / "split_summary.csv",
    index=False,
    encoding="utf-8",
)

original_reference = pd.DataFrame(
    [
        {
            "strategy": "original",
            "seed": np.nan,
            "total_examples": len(df),
            "total_pairs": df["pair_id"].nunique(),
            "train_examples": EXPECTED_SPLIT_COUNTS["train"],
            "validation_examples": EXPECTED_SPLIT_COUNTS["validation"],
            "test_examples": EXPECTED_SPLIT_COUNTS["test"],
            "cross_split_pairs": original_cross_split_pairs,
            "cross_split_rate": original_cross_split_rate,
        }
    ]
)

comparison_df = pd.concat(
    [
        original_reference,
        all_summary_df,
    ],
    ignore_index=True,
)

comparison_df.to_csv(
    DATA_DIR / "split_comparison.csv",
    index=False,
    encoding="utf-8",
)

display(comparison_df)

,strategy,seed,total_examples,total_pairs,train_examples,validation_examples,test_examples,cross_split_pairs,cross_split_rate
0,original,NaN,5700,2850,3990,570,1140,1306,0.458246
1,pair_controlled,13.0,5700,2850,3990,570,1140,0,0.000000
2,pair_controlled,21.0,5700,2850,3990,570,1140,0,0.000000
3,pair_controlled,40.0,5700,2850,3990,570,1140,0,0.000000
4,pair_controlled,42.0,5700,2850,3990,570,1140,0,0.000000
5,pair_controlled,73.0,5700,2850,3990,570,1140,0,0.000000
6,max_cross_split,13.0,5700,2850,3990,570,1140,1710,0.600000
7,max_cross_split,21.0,5700,2850,3990,570,1140,1710,0.600000
8,max_cross_split,40.0,5700,2850,3990,570,1140,1710,0.600000
9,max_cross_split,42.0,5700,2850,3990,570,1140,1710,0.600000


In [26]:
saved_file_checks = []

for strategy in STRATEGIES:
    for seed in SPLIT_SEEDS:
        seed_dir = (
            OUTPUT_DIRS[strategy]
            / f"seed_{seed}"
        )

        row = {
            "strategy": strategy,
            "seed": seed,
        }

        for split_name in SPLIT_NAMES:
            path = (
                seed_dir
                / f"{split_name}.jsonl"
            )

            if not path.is_file():
                raise FileNotFoundError(
                    f"Missing generated file: {path}"
                )

            observed_lines = count_jsonl_lines(
                path
            )

            expected_lines = (
                EXPECTED_SPLIT_COUNTS[
                    split_name
                ]
            )

            if observed_lines != expected_lines:
                raise ValueError(
                    f"{strategy}/{seed}/{split_name}: expected {expected_lines}, found {observed_lines}."
                )

            row[
                f"{split_name}_lines"
            ] = observed_lines

        saved_file_checks.append(row)

saved_file_checks_df = pd.DataFrame(
    saved_file_checks
)

display(saved_file_checks_df)

,strategy,seed,train_lines,validation_lines,test_lines
0,pair_controlled,13,3990,570,1140
1,pair_controlled,21,3990,570,1140
2,pair_controlled,40,3990,570,1140
3,pair_controlled,42,3990,570,1140
4,pair_controlled,73,3990,570,1140
5,max_cross_split,13,3990,570,1140
6,max_cross_split,21,3990,570,1140
7,max_cross_split,40,3990,570,1140
8,max_cross_split,42,3990,570,1140
9,max_cross_split,73,3990,570,1140


In [27]:
print(
    "original",
    original_cross_split_pairs,
    f"{original_cross_split_rate:.4%}",
)

for strategy in STRATEGIES:
    strategy_rows = all_summary_df[
        all_summary_df["strategy"] == strategy
    ]

    print(
        strategy,
        strategy_rows["cross_split_pairs"].tolist(),
        strategy_rows["cross_split_rate"].tolist(),
    )

original 1306 45.8246%
pair_controlled [0, 0, 0, 0, 0] [0.0, 0.0, 0.0, 0.0, 0.0]
max_cross_split [1710, 1710, 1710, 1710, 1710] [0.6, 0.6, 0.6, 0.6, 0.6]
